# CLIP + SAM Zero-Shot Visual Grounding

This notebook demonstrates the full pipeline:
1. Load SAM + CLIP models
2. Generate masks with SAM
3. Filter masks by area
4. Score masks with CLIP (multi-prompt + area penalty)
5. Visualize the best matching mask

In [ ]:
import sys
sys.path.append('..')  # so src/ imports work from notebooks/

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

## 1. Load Models

In [ ]:
from src.clip_utils import load_clip_model
from src.sam_utils import load_sam_model

print('Loading CLIP...')
clip_model, clip_processor = load_clip_model(device='cpu')
print('CLIP loaded ✓')

print('Loading SAM...')
sam = load_sam_model(
    checkpoint_path='../checkpoints/sam_vit_b_01ec64.pth',
    model_type='vit_b',
    device='cpu',
)
print('SAM loaded ✓')

## 2. Load & Inspect Image

In [ ]:
IMAGE_PATH = '../data/54cae422a6fc6_-_american-muscle-facts-03-0312-xln.jpg'
PROMPT = 'car'

image_bgr = cv2.imread(IMAGE_PATH)
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Resize to max 800px wide preserving aspect ratio
h, w = image.shape[:2]
if w > 800:
    scale = 800 / w
    image = cv2.resize(image, (800, int(h * scale)))

print(f'Image shape: {image.shape}')

plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title('Input Image')
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Generate Masks with SAM

In [ ]:
from src.sam_utils import generate_masks, filter_masks

masks = generate_masks(image=image, sam=sam, points_per_side=32)
print(f'Total masks generated: {len(masks)}')

# Show mask area distribution
areas = [m['area'] for m in masks]
image_area = image.shape[0] * image.shape[1]

plt.figure(figsize=(8, 3))
plt.hist([a / image_area for a in areas], bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Mask area (fraction of image)')
plt.ylabel('Count')
plt.title('Mask Area Distribution')
plt.tight_layout()
plt.show()

## 4. Filter Masks by Area

In [ ]:
filtered_masks = filter_masks(
    masks=masks,
    image_area=image_area,
    min_area_ratio=0.01,
    max_area_ratio=0.95,
)

print(f'Masks after filtering: {len(filtered_masks)}')

## 5. Ground Prompt with CLIP

In [ ]:
from src.grounding import find_best_mask

print(f'Prompt: "{PROMPT}"')
print()

best_idx, best_score, all_scores = find_best_mask(
    image=image,
    masks=filtered_masks,
    prompt=PROMPT,
    model=clip_model,
    processor=clip_processor,
    device='cpu',
)

print(f'Best mask index : {best_idx}')
print(f'Confidence      : {best_score:.1%}')
print(f'Mask area ratio : {filtered_masks[best_idx]["area"] / image_area:.2%}')

## 6. Visualize Top Results

In [ ]:
from src.visualization import create_visualization

result = create_visualization(
    image=image,
    masks=filtered_masks,
    best_idx=best_idx,
    prompt=PROMPT,
    best_score=best_score,
)

plt.figure(figsize=(14, 6))
plt.imshow(result)
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Score Table — Top 10 Masks

In [ ]:
print(f'{"Rank":<6} {"Idx":<6} {"Confidence":<12} {"Area Ratio":>10}')
print('-' * 40)

for rank, (idx, score) in enumerate(all_scores[:10], 1):
    area_r = filtered_masks[idx]['area'] / image_area
    print(f'{rank:<6} {idx:<6} {score:<12.1%} {area_r:>9.2%}')